# Fine-tuning de LLM para cuidado de plantas

Este notebook entrena un modelo de lenguaje (Phi-3-mini) con **Unsloth** y **LoRA** sobre un dataset de instrucciones de cuidado de plantas (luz, riego, poda), para que el modelo aprenda a dar recomendaciones específicas por especie.

## 1. Instalación de dependencias

Instalamos **Unsloth** (entrenamiento más rápido y eficiente en memoria), **TRL** (SFTTrainer), **PEFT** (LoRA) y **bitsandbytes** (cuantización 4-bit). En Colab conviene ejecutar esta celda primero.

In [ ]:
!pip uninstall -y unsloth peft
!pip install unsloth trl peft accelerate bitsandbytes

Found existing installation: peft 0.18.1
Uninstalling peft-0.18.1:
  Successfully uninstalled peft-0.18.1
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.8/62.8 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.0/557.0 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 19.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 55.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.2/403.2 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 20.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━

## 2. Imports y comprobación de GPU

- **pandas**: manejo del CSV de plantas.  
- **torch**: comprobar si hay CUDA disponible.  
- **FastLanguageModel** (Unsloth): carga y fine-tuning eficiente del modelo.

In [ ]:
import pandas as pd
import torch
from unsloth import FastLanguageModel

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


## 3. Comprobar GPU

Verificamos que PyTorch vea la GPU (por ejemplo, T4 en Colab). Sin GPU el entrenamiento sería muy lento.

In [ ]:
print(f"CUDA disponibles: {torch.cuda.is_available()}")
print(f"GP {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

CUDA disponibles: True
GP Tesla T4


## 4. Carga del dataset de plantas

Leemos el CSV con información por planta: **name**, **scientific_name**, **sunlight**, **watering**, **pruning**. El archivo debe estar en el directorio de trabajo (o indicar la ruta correcta).

In [ ]:
file = pd.read_csv(open("cuidados_plantas.csv", "r"), sep=",")

file.head()

,id,species_id,name,scientific_name,sunlight,watering,pruning
0,1,352,Ambrosia Apple,"[""Malus 'Ambrosia'""]",Ambrosia Apple (Malus 'Ambrosia') needs at lea...,Ambrosia Apple (Malus 'Ambrosia') plants requi...,The Ambrosia Apple (Malus 'Ambrosia') should b...
1,4,367,Honeycrisp Apple,"[""Malus 'Honeycrisp'""]","Sunlight is essential for all types of apples,...",Honeycrisp Apple (Malus 'Honeycrisp') should b...,Pruning Honeycrisp apples is an essential main...
2,5,1893,mandarin orange,"[""Citrus reticulata 'Clementine'""]",Mandarin orange plants should be pruned annual...,"Generally, mandarin orange trees need about on...",Mandarin oranges need plenty of sunlight to gr...
3,6,1895,navel orange,"[""Citrus sinensis 'Trovita'""]",Navel oranges need a considerable amount of su...,Water your navel orange tree (Citrus sinensis ...,Because Navel orange (Citrus sinensis 'Trovita...
4,7,1896,navel orange,"[""Citrus sinensis 'Washington'""]",The navel orange tree likes plenty of sunlight...,Navel oranges need to be watered 1–2 times per...,Naval orange trees should be pruned around the...


## 5. Carga del modelo base (Phi-3)

- **Modelo**: `Phi-3-mini-4k-instruct` en 4-bit para reducir uso de VRAM.  
- **max_seq_length**: longitud máxima de secuencia (2048).  
- **load_in_4bit=True**: cuantización 4-bit para poder entrenar en GPUs con poca memoria.

In [ ]:
model_name = "unsloth/Phi-3-mini-4k-instruct-bnb-4bit" # este es un modelo muy muy pequeño, para que dure poco el entrenamiento
max_seq_length = 2048
dtype = None # Auto detection

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=True
)

==((====))==  Unsloth 2026.3.17: Fast Mistral patching. Transformers: 5.3.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.26G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/194 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/458 [00:00<?, ?B/s]

## 6. Formateo del dataset para entrenamiento

Convertimos cada fila del CSV al formato de chat del modelo:  
`<|user|>` + pregunta (nombre común y científico) + `<|assistant|>` + respuesta (Sunlight, Watering, Pruning).  
Así el modelo aprende a generar instrucciones de cuidado dadas las entradas.

In [ ]:
def format_prompt(dataset) -> list:
    formatted = []

    for _, plant in dataset.iterrows():
        text = (
            "<|user|>\n"
            "Provide complete care instructions for the following plant.\n\n"
            f"Common name: {plant['name']}\n"
            f"Scientific name: {plant['scientific_name']}\n"
            "<|assistant|>\n"
            f"Sunlight: {plant['sunlight']}\n"
            f"Watering: {plant['watering']}\n"
            f"Pruning: {plant['pruning']}"
        )

        formatted.append({"text": text})

    return formatted


formatted_data = format_prompt(file)


## 7. Configuración LoRA (Parameter-Efficient Fine-Tuning)

Aplicamos **LoRA** solo a parte de las capas (attention + MLP) para entrenar pocos parámetros y ahorrar memoria y tiempo.  
- **r=16**: rango de LoRA.  
- **target_modules**: q_proj, k_proj, v_proj, o_proj, gate_proj, up_proj, down_proj.  
- **use_gradient_checkpointing="unsloth"**: reduce VRAM durante el entrenamiento.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth 2026.3.17 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


## 8. Entrenamiento con SFTTrainer

- **SFTTrainer**: Supervised Fine-Tuning sobre el dataset formateado.  
- **packing=True**: empaqueta varias muestras en una secuencia para aprovechar mejor la longitud máxima (recomendado para Phi-3).  
- **batch_size**, **gradient_accumulation_steps**, **num_train_epochs**, **learning_rate**: hiperparámetros de entrenamiento.  
- **output_dir**: carpeta donde se guardan checkpoints (p. ej. `./outputs`).

In [ ]:
from datasets import Dataset
from trl import SFTTrainer, SFTConfig

train_dataset = Dataset.from_list(formatted_data)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=4096,
        packing=True,          # MUY recomendable para Phi-3
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        num_train_epochs=3,
        learning_rate=2e-4,
        fp16=True,
        logging_steps=10,
        output_dir="./outputs",
    ),
)


Unsloth: You set `max_seq_length` as 4096 but the maximum the model supports is 2048. We shall reduce it.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/3000 [00:00<?, ? examples/s]

Unsloth: Packing train dataset (num_proc=6):   0%|          | 0/3000 [00:00<?, ? examples/s]

🦥 Unsloth: Packing enabled - training is >2x faster and uses less VRAM!


In [ ]:
# 10.1) Exportación para API
# Mantener la carpeta LoRA
trainer.model.save_pretrained("./phi3-plant-care-lora")
tokenizer.save_pretrained("./phi3-plant-care-lora")

# Opcional: generar un artefacto .keras cuando la arquitectura TensorFlow esté disponible
# (en algunos modelos LLM puede no existir implementación TF compatible)
try:
    from transformers import TFAutoModelForCausalLM

    tf_model = TFAutoModelForCausalLM.from_pretrained(
        "./phi3-plant-care-lora",
        from_pt=True,
    )
    tf_model.save("./phi3-plant-care-lora.keras")
    print("Modelo exportado en: ./phi3-plant-care-lora.keras")
except Exception as e:
    print("No se pudo exportar a .keras automáticamente.")
    print(f"Detalle: {e}")
    print("Se mantiene el formato Hugging Face en ./phi3-plant-care-lora para usarlo en la API.")

No se pudo exportar a .keras automáticamente.
Detalle: cannot import name 'TFAutoModelForCausalLM' from 'transformers' (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)
Se mantiene el formato Hugging Face en ./phi3-plant-care-lora para usarlo en la API.


In [ ]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 687 | Num Epochs = 3 | Total steps = 258
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,884,416 of 3,850,963,968 (0.78% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.543558
20,1.379198
30,1.166208
40,1.123048
50,1.110173
60,1.100921
70,1.105490
80,1.089029
90,1.080030
100,1.067538


TrainOutput(global_step=258, training_loss=1.1022827643756719, metrics={'train_runtime': 6288.3287, 'train_samples_per_second': 0.328, 'train_steps_per_second': 0.041, 'total_flos': 8.920270343227392e+16, 'train_loss': 1.1022827643756719, 'epoch': 3.0})

In [ ]:
!zip -r phi3-plant-care-lora.zip phi3-plant-care-lora
from google.colab import files
files.download("phi3-plant-care-lora.zip")

  adding: phi3-plant-care-lora/ (stored 0%)
  adding: phi3-plant-care-lora/README.md (deflated 65%)
  adding: phi3-plant-care-lora/tokenizer_config.json (deflated 45%)
  adding: phi3-plant-care-lora/chat_template.jinja (deflated 60%)
  adding: phi3-plant-care-lora/adapter_config.json (deflated 58%)
  adding: phi3-plant-care-lora/adapter_model.safetensors (deflated 8%)
  adding: phi3-plant-care-lora/tokenizer.json (deflated 85%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
trainer.model.save_pretrained("./phi3-plant-care-lora")
tokenizer.save_pretrained("./phi3-plant-care-lora")


('./phi3-plant-care-lora/tokenizer_config.json',
 './phi3-plant-care-lora/chat_template.jinja',
 './phi3-plant-care-lora/tokenizer.json')

## 10. Guardar el modelo y el tokenizer

Guardamos los adaptadores LoRA y el tokenizer en `./phi3-plant-care-lora`. Para usarlos después hay que cargar el modelo base y aplicar estos pesos (o usar `from_pretrained` sobre esa carpeta si el código lo soporta).

## 11. Probar el modelo

A continuación se prepara el modelo para inferencia y se genera una respuesta de ejemplo con el mismo formato de prompt usado en el entrenamiento.

In [ ]:
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): MistralForCausalLM(
      (model): MistralModel(
        (embed_tokens): Embedding(32064, 3072, padding_idx=32009)
        (layers): ModuleList(
          (0-31): 32 x MistralDecoderLayer(
            (self_attn): MistralAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=3072, out_features=3072, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=3072, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=3072, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj

**Preparar para inferencia**  
`FastLanguageModel.for_inference(model)` optimiza el modelo para generar texto (desactiva dropout, etc.).

In [ ]:
prompt = (
    "<|user|>\n"
    "Provide complete care instructions for the following plant.\n\n"
    "Common name: Ambrosia Apple\n"
    "Scientific name: Malus Ambrosia\n"
    "<|assistant|>\n"
)

inputs = tokenizer(
    prompt,
    return_tensors="pt",
).to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=256,
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=False))


Both `max_new_tokens` (=256) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


RuntimeError: output with shape [1, 32, 1, 96] doesn't match the broadcast shape [1, 32, 34, 96]

**Generar respuesta**  
Mismo formato que en entrenamiento: `<|user|>` + pregunta por planta + `<|assistant|>`.  
Se usan `temperature` y `top_p` para controlar la diversidad del texto generado.